# 충남대 학내정보 챗봇 - 5-way Question Classifier

0: 졸업요건 / 1: 공지 / 2: 학사일정 / 3: 식단 / 4: 셔틀

- Model: KLUE-RoBERTa-small fine-tuned
- 평가 시 이 노트북이 실행됨. `model/model.bin`이 있으면 추론만, 없으면 학습 후 추론.
- Input:  `data/test_cls.json`  (비공개 평가셋)
- Output: `outputs/cls_output.json`


## 0. Setup


In [ ]:
import os, sys, json, subprocess
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))) if os.path.basename(os.getcwd()) == 'src' else os.path.abspath(os.getcwd())
# Heuristic: if cwd is the project root, use it; if cwd is `src/`, go up one
if os.path.basename(os.getcwd()) == 'src':
    ROOT = os.path.dirname(os.getcwd())
else:
    ROOT = os.getcwd()
print('ROOT =', ROOT)
sys.path.insert(0, os.path.join(ROOT, 'src'))


In [ ]:
# Required packages (Colab/평가 환경에 없을 때만 설치)
try:
    import transformers, torch
    print('transformers', transformers.__version__, 'torch', torch.__version__)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers==4.46.0', 'torch==2.5.1'], check=True)
    import transformers, torch


## 1. 학습 (model.bin 없을 때만)


In [ ]:
MODEL_DIR = os.path.join(ROOT, 'model')
MODEL_BIN = os.path.join(MODEL_DIR, 'model.bin')
TRAIN_JSONL = os.path.join(ROOT, 'data', 'classifier', 'train.jsonl')

need_train = not os.path.exists(MODEL_BIN)
if need_train and not os.path.exists(TRAIN_JSONL):
    raise FileNotFoundError(
        f'학습 데이터({TRAIN_JSONL})도 model.bin도 없습니다. '
        '제출 시 model/model.bin을 함께 제출해야 합니다.')

if need_train:
    print('[notebook] model.bin not found - training from scratch...')
    from classifier.train import train_loop
    import argparse
    args = argparse.Namespace(
        model_name='klue/roberta-small', epochs=6, batch_size=32,
        lr=2e-5, max_len=64, seed=2026,
    )
    train_loop(args)
else:
    print('[notebook] model.bin found - skipping training.')


## 2. 추론: test_cls.json -> cls_output.json


In [ ]:
from classifier.predict import run as predict_run
INPUT  = os.path.join(ROOT, 'data', 'test_cls.json')
OUTPUT = os.path.join(ROOT, 'outputs', 'cls_output.json')

if not os.path.exists(INPUT):
    # Fallback: D-1 eval set으로 sanity check
    INPUT = os.path.join(ROOT, 'data', 'classifier', 'd1_eval.jsonl')
    print(f'[notebook] test_cls.json 없음 -> D-1 eval set으로 fallback ({INPUT})')
    # Convert d1_eval.jsonl to mock test_cls.json schema
    mock = []
    with open(INPUT, 'r', encoding='utf-8') as f:
        for line in f:
            x = json.loads(line)
            mock.append({'id': x['qa_id'], 'question': x['text']})
    INPUT = os.path.join(ROOT, 'data', '_d1_as_test.json')
    with open(INPUT, 'w', encoding='utf-8') as f:
        json.dump(mock, f, ensure_ascii=False, indent=2)

predict_run(INPUT, OUTPUT, MODEL_DIR)
print('Done. Output saved to', OUTPUT)


## 3. (선택) 결과 미리보기


In [ ]:
with open(OUTPUT, 'r', encoding='utf-8') as f:
    preds = json.load(f)
print(f'총 {len(preds)}건 예측')
print('샘플 5건:')
for p in preds[:5]:
    print(' ', p)
from collections import Counter
print('라벨 분포:', dict(Counter(p["label"] for p in preds)))
